In [ ]:
"""
知识点
：
1.
-> bool 是 Python 类型注解（Type Hint）
含义："这个方法返回值应该是布尔类型
def is_borrowed(self) -> bool:
    return self._is_borrowed
#              ↑
#         告诉你（和IDE）：这个函数返回的是 bool 类型
参数注解	name: str	name 应该是字符串
返回值注解	-> bool	返回值应该是布尔类型
关键：只是"提示"，不强制！



2.
__str__ —— 给用户看的
python
def __str__(self):
    status = "📕 已借出" if self._is_borrowed else "📗 在馆"
    return f"[{status}] 《{title}》 - {author} (ISBN:{isbn})".format(
        title=self.title, author=self.author, isbn=self.isbn
    )

触发时机：print(book) 或 str(book) 时自动调用
目标读者：终端用户，要求可读、友好
效果示例：[📗 在馆] 《Python编程从入门到实践》 - Eric Matthes (ISBN:978-7-115-)

__repr__ —— 给开发者看的
python
def __repr__(self):
    return f"Book('{self.title}', '{self.author}', '{self.isbn}')"

触发时机：在交互式终端直接输入变量名、或调试时自动调用
目标读者：开发者，要求能还原对象
效果示例：Book('Python编程从入门到实践', 'Eric Matthes', '978-7-115-')
理想标准：返回的字符串应该能让 eval(repr(obj)) 重建对象（这里简化了）


对比
                 __str__	                          __repr__
调用方式	    print(obj) / str(obj)	        直接输入 obj 或 repr(obj)
面向谁	    用户（好看）	                    开发者（准确）
类比	        名片	                        身份证

简单记忆
__str__ = 字符串化（String），__repr__ = 表示（Representation）
"""

In [15]:
class BookNotFoundError(Exception):
    """
    图书不存在异常
    """
    pass
class BookIsBorrowedError(Exception):
    """"
    图书已借出异常
    """
    pass
class BookIsInLibraryError(Exception):
    """
    图书已在库异常
    """
    pass

class Book:
    def __init__(self,title:str,author:str,isbn:str):
        self.title = title
        self.author = author
        self.isbn = isbn
        self._is_borrowed = False

    @property
    def is_borrowed(self)->bool:
        return self._is_borrowed

    def borrowed(self):
        if self.is_borrowed:
            raise BookIsBorrowedError(f"'{self.title}' 已被借出")
        self._is_borrowed = True

    def return_book(self):
        if not self.is_borrowed:
            raise BookIsInLibraryError(f"'{self.title}' 未被借出，无法归还")
        self._is_borrowed = False

    def __str__(self):
        status = "📕 已借出" if self._is_borrowed else "📗 在馆"
        return f"[{status}] 《{self.title}》 - {self.author} (ISBN:{self.isbn})"

    def __repr__(self):
        return f"Book('{self.title}', '{self.author}', '{self.isbn}')"


class Library:
    #    str="我的图书馆”     默认参数
    def __init__(self, name:str="我的图书馆"):
        self.name = name
        self._books = {}

    def add_book(self,book:Book):
        if book.isbn in self._books:
            raise BookIsInLibraryError(f"ISBN {book.isbn} ('{book.title}') 已存在")
        self._books[book.isbn] = book
        print(f"✅ 已添加: {book.title}")

    def borrow_book(self,isbn:str):
        if isbn not in self._books:
            raise BookNotFoundError(f"找不到 ISBN：{isbn}")
        book = self._books[isbn]
        book.borrowed()
        print(f"📤 已借出: {book.title}")
        return book

    def return_book(self,isbn:str):
        if isbn not in self._books:
            raise BookNotFoundError(f"找不到 ISBN: {isbn}")
        book = self._books[isbn]
        book.return_book()
        print(f"📥 已归还: {book.title}")


    def list_books(self):
        """列出所有书籍及其状态"""
        if not self._books:
            print("📭 图书馆空空如也...")
            return

        print(f"\n{'='*50}")
        print(f"📚 {self.name} - 共 {len(self._books)} 本")
        print('='*50)

        available = []
        borrowed = []

        for book in self._books.values():
            if book.is_borrowed:
                borrowed.append(book)
            else:
                available.append(book)

        if available:
            print("\n📗 在馆书籍:")
            for book in available:
                print(f"   {book}")

        if borrowed:
            print("\n📕 已借出:")
            for book in borrowed:
                print(f"   {book}")

    def search(self, keyword: str) -> list:
        """
        按关键词搜索（匹配书名或作者）

        Returns:
            匹配的 Book 列表
        """
        keyword = keyword.lower()
        results = [
            book for book in self._books.values()
            if keyword in book.title.lower() or keyword in book.author.lower()
        ]

        if results:
            print(f"\n🔍 搜索 '{keyword}' 找到 {len(results)} 本:")
            for book in results:
                print(f"   {book}")
        else:
            print(f"\n🔍 搜索 '{keyword}' 无结果")

        return results



if __name__ == "__main__":
    lib = Library("南林大图书馆")

    # 添加书籍
    books = [
        Book("Python编程从入门到实践", "Eric Matthes", "978-7-115-1"),
        Book("流畅的Python", "Luciano Ramalho", "978-7-115-2"),
        Book("深入理解计算机系统", "Randal E.Bryant", "978-7-111-3"),
    ]

    for book in books:
        lib.add_book(book)

    # 列出所有书
    lib.list_books()

    # 借书
    print("\n--- 借书测试 ---")
    lib.borrow_book("978-7-115-1")  # 借第一本

    # # 再次借同一本（应报错）
    # try:
    #     lib.borrow_book("978-7-115-")
    # except BookIsBorrowedError as e:
    #     print(f"❌ {e}")

    # 搜索
    lib.search("python")
    lib.list_books()

    # 还书
    print("\n--- 还书测试 ---")
    lib.return_book("978-7-115-1")
    lib.list_books()

✅ 已添加: Python编程从入门到实践
✅ 已添加: 流畅的Python
✅ 已添加: 深入理解计算机系统

📚 南林大图书馆 - 共 3 本

📗 在馆书籍:
   [📗 在馆] 《Python编程从入门到实践》 - Eric Matthes (ISBN:978-7-115-1)
   [📗 在馆] 《流畅的Python》 - Luciano Ramalho (ISBN:978-7-115-2)
   [📗 在馆] 《深入理解计算机系统》 - Randal E.Bryant (ISBN:978-7-111-3)

--- 借书测试 ---
📤 已借出: Python编程从入门到实践

🔍 搜索 'python' 找到 2 本:
   [📕 已借出] 《Python编程从入门到实践》 - Eric Matthes (ISBN:978-7-115-1)
   [📗 在馆] 《流畅的Python》 - Luciano Ramalho (ISBN:978-7-115-2)

📚 南林大图书馆 - 共 3 本

📗 在馆书籍:
   [📗 在馆] 《流畅的Python》 - Luciano Ramalho (ISBN:978-7-115-2)
   [📗 在馆] 《深入理解计算机系统》 - Randal E.Bryant (ISBN:978-7-111-3)

📕 已借出:
   [📕 已借出] 《Python编程从入门到实践》 - Eric Matthes (ISBN:978-7-115-1)

--- 还书测试 ---
📥 已归还: Python编程从入门到实践

📚 南林大图书馆 - 共 3 本

📗 在馆书籍:
   [📗 在馆] 《Python编程从入门到实践》 - Eric Matthes (ISBN:978-7-115-1)
   [📗 在馆] 《流畅的Python》 - Luciano Ramalho (ISBN:978-7-115-2)
   [📗 在馆] 《深入理解计算机系统》 - Randal E.Bryant (ISBN:978-7-111-3)
